# Qwen2.5-7B-Instruct — Product Name Extraction — **Notebook 1: Fine-tune**
### Fine-tune + out-of-sample test, then download the LoRA adapter (no Unsloth)

Built for the **NVIDIA RTX PRO 6000 Blackwell (96 GB, sm_120)**.

This is **notebook 1 of 2**. It fine-tunes Qwen2.5-7B-Instruct with LoRA, runs the out-of-sample test, and **downloads the trained LoRA adapter to your PC** (~160 MB). If Colab disconnects during inference you never have to re-train: just run **notebook 2**, upload this adapter, and go straight to inference.

**Why save only the adapter:** LoRA training leaves the base weights frozen, so the only thing learned is the small adapter (r=16 on 7 modules ≈ 40M params). Saved in fp32 that is ~160 MB, versus ~30 GB for the full merged fp32 model. Re-attaching the adapter to the base model in notebook 2 is deterministic, so **exact reproducibility is preserved**.

**Sections:** 1) install · 2) reproducibility · 3–6) load labeled data · 7–9) fine-tune with per-epoch loss table · 10) out-of-sample test · **12) save + download the adapter**.

## 1. Install dependencies (exact pinned versions — restarts once)

Installs the **exact** versions requested and then **restarts the session automatically** so the new `torch`/`numpy` binaries are the ones actually imported.

**How to run:** run this cell once → it installs everything and restarts the kernel → when it comes back, **re-run this same cell** (it detects a sentinel file and skips the reinstall) → then run cell **1b** to confirm every version.

Target stack: torch 2.10.0+cu128 / torchvision 0.25.0 / torchaudio 2.10.0, numpy 2.0.2, scipy 1.16.3, scikit-learn 1.6.1, pandas 2.2.2, transformers 4.55.0, trl 0.20.0, peft 0.16.0, accelerate 1.9.0, datasets 3.6.0, openpyxl 3.1.5, sentencepiece 0.2.1.

In [ ]:
# --- Install EXACT pinned versions, then restart the session once ------------
# Target stack (all versions pinned exactly, per request):
#     torch 2.10.0+cu128 / torchvision 0.25.0 / torchaudio 2.10.0  (Blackwell sm_120)
#     numpy 2.0.2 | scipy 1.16.3 | scikit-learn 1.6.1 | pandas 2.2.2
#     transformers 4.55.0 | trl 0.20.0 | peft 0.16.0 | accelerate 1.9.0
#     datasets 3.6.0 | openpyxl 3.1.5 | sentencepiece 0.2.1
#
# Because we change torch and numpy (vs. what Colab preinstalls), the kernel
# MUST restart once so the new binaries are the ones actually imported. This
# cell installs everything, then restarts automatically. After the restart,
# just RE-RUN this cell: it detects the sentinel file and skips reinstalling,
# then falls through to the next cell (1b) for verification.

import os, sys, subprocess

SENTINEL = "/content/.install_done_v3"   # bump this string if you change pins

def sh(cmd):
    print(">>", cmd, flush=True)
    # no -q: full pip output is shown
    subprocess.run(cmd, shell=True, check=False)

if os.path.exists(SENTINEL):
    print("Sentinel found — packages already installed in this session.")
    print("Skipping reinstall. Continue to cell 1b to verify versions.")
else:
    # 1) Torch stack from the cu128 index (Blackwell sm_120 kernels).
    #    torchvision/torchaudio are pinned to the versions that match torch 2.10.0.
    sh(f'{sys.executable} -m pip install '
       f'torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 '
       f'--index-url https://download.pytorch.org/whl/cu128')

    # 2) Scientific stack from PyPI, pinned exactly.
    sh(f'{sys.executable} -m pip install '
       f'"numpy==2.0.2" "scipy==1.16.3" "scikit-learn==1.6.1" "pandas==2.2.2"')

    # 3) Hugging Face training stack + IO libs.
    #    trl 0.20.0 requires transformers>=4.55.0 / accelerate>=1.4.0 / datasets>=3.0.0
    #    -> satisfied by the pins below.
    sh(f'{sys.executable} -m pip install '
       f'"transformers==4.55.0" "trl==0.20.0" "peft==0.16.0" "accelerate==1.9.0" '
       f'"datasets==3.6.0" "openpyxl==3.1.5" "sentencepiece==0.2.1"')

    # Mark done so the post-restart re-run skips the installs above.
    with open(SENTINEL, "w") as f:
        f.write("ok")

    print("\n" + "="*70)
    print("Install complete. RESTARTING the session now so the new torch/numpy")
    print("binaries are the ones imported. After it restarts, RE-RUN THIS CELL")
    print("(it will skip reinstalling), then run cell 1b to verify versions.")
    print("="*70, flush=True)

    # Trigger the restart. (Colab shows 'Your session crashed/restarted' — this
    # is expected and intentional, not an error.)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)


### 1b. Verify exact versions

Asserts that the pinned packages match exactly (torch 2.10.0+cu128, torchvision 0.25.0, torchaudio 2.10.0, numpy 2.0.2, scipy 1.16.3, scikit-learn 1.6.1, pandas 2.2.2, transformers 4.55.0, trl 0.20.0, peft 0.16.0, accelerate 1.9.0, datasets 3.6.0, openpyxl 3.1.5, sentencepiece 0.2.1), that torch is a `cu128` build with `sm_120` (Blackwell) kernels, and that `sklearn` imports. Run this **after** the install cell has restarted and been re-run.

In [ ]:
# --- Verify the pinned versions are the ones actually loaded ----------------
import platform, importlib

# All packages pinned to exact versions.
EXPECTED_EXACT = {
    "torch"        : "2.10.0",     # checked on leading x.y.z (build tag +cu128 follows)
    "torchvision"  : "0.25.0",
    "torchaudio"   : "2.10.0",
    "numpy"        : "2.0.2",
    "scipy"        : "1.16.3",
    "sklearn"      : "1.6.1",      # scikit-learn imports as "sklearn"
    "pandas"       : "2.2.2",
    "transformers" : "4.55.0",
    "trl"          : "0.20.0",
    "peft"         : "0.16.0",
    "accelerate"   : "1.9.0",
    "datasets"     : "3.6.0",
    "openpyxl"     : "3.1.5",
    "sentencepiece": "0.2.1",
}
EXPECTED_MIN = {}  # (openpyxl / sentencepiece now pinned exactly above)

def _tuple(v):
    parts = v.split("+")[0].split(".")
    out = []
    for p in parts:
        try: out.append(int(p))
        except ValueError: out.append(0)
    return tuple(out)

print(f"Python : {platform.python_version()}  (expected 3.12.13)\n")

problems = []
for mod, want in EXPECTED_EXACT.items():
    try:
        m = importlib.import_module(mod)
        got = getattr(m, "__version__", "?")
        ok = got.split("+")[0] == want
        print(f"{'OK ' if ok else '!! '}{mod:14s} {got:20s} (expected {want})")
        if not ok:
            problems.append(f"{mod}: got {got}, expected {want}")
    except Exception as e:
        print(f"!! {mod:14s} IMPORT FAILED: {e}")
        problems.append(f"{mod}: import failed ({e})")

for mod, want in EXPECTED_MIN.items():
    try:
        m = importlib.import_module(mod)
        got = getattr(m, "__version__", "?")
        ok = _tuple(got) >= want
        print(f"{'OK ' if ok else '!! '}{mod:14s} {got:20s} (expected >= {'.'.join(map(str,want))})")
        if not ok:
            problems.append(f"{mod}: got {got}, expected >= {'.'.join(map(str,want))}")
    except Exception as e:
        print(f"!! {mod:14s} IMPORT FAILED: {e}")
        problems.append(f"{mod}: import failed ({e})")

# CUDA build tag + Blackwell kernels
import torch
print(f"\ntorch CUDA build: {torch.version.cuda}  (expected 12.8)")
if "cu128" not in (torch.__version__ or ""):
    problems.append(f"torch is not a +cu128 build: {torch.__version__}")

# sklearn smoke test (the exact import Section 5 needs)
from sklearn.model_selection import train_test_split
print("sklearn import OK — train_test_split available.")

if torch.cuda.is_available():
    print("\nGPU :", torch.cuda.get_device_name(0),
          "| cc", torch.cuda.get_device_capability(0))
    archs = torch.cuda.get_arch_list()
    print("arch list:", archs)
    if not any("sm_120" in a or "sm_100" in a for a in archs):
        problems.append("torch has NO Blackwell (sm_120) kernels")
    else:
        print("OK: Blackwell (sm_120) kernels present.")
else:
    problems.append("no CUDA GPU detected")

print("\n" + "="*60)
if problems:
    print("VERSION/ENV PROBLEMS DETECTED:")
    for p in problems:
        print("  -", p)
    raise SystemExit(
        "Environment does not match the pinned target. If you just ran the "
        "install cell, make sure you RE-RAN it after the restart. Otherwise "
        "re-run Section 1, let it restart, then re-run it once more."
    )
else:
    print("All versions match the pinned target. Continue to the data cells.")


## 2. Global reproducibility setup (full fp32, hard determinism)

In [ ]:
import os, random, numpy as np, torch

SEED = 42

# --- Deterministic env flags: set BEFORE any CUDA kernels initialize ---------
os.environ["PYTHONHASHSEED"]          = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # required for deterministic cuBLAS matmul

def set_all_seeds(seed: int = SEED, hard: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    # TF32 rounds fp32 matmuls to ~10 bits nondeterministically -> keep OFF for exact fp32.
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    # hard=True: raise if ANY op lacks a deterministic implementation (true guarantee).
    # We train in fp32 with stock transformers ops, which all HAVE deterministic kernels,
    # so hard mode should NOT crash here (unlike with Unsloth's fused kernels).
    torch.use_deterministic_algorithms(hard, warn_only=not hard)

set_all_seeds(SEED, hard=True)
print("Seeds + HARD determinism set. CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("\nEXACT-REPRODUCIBILITY MODE: full fp32 compute (no quantization, no 16-bit),")
print("deterministic cuBLAS/cuDNN, TF32 off, fixed seeds. Same GPU -> identical losses.")


## 3. Upload the labeled fine-tuning file

Upload only `Announcements manually labeled fine tuning sample.xlsx` (columns: `announcement`, `products`). The inference chunks are uploaded later, one at a time, in **Section 11**.

In [ ]:
from google.colab import files

# Local working directory on the Colab VM (not Drive)
WORK_DIR   = "/content/work"
OUTPUT_DIR = os.path.join(WORK_DIR, "qwen25_7b_lora_products")   # LoRA adapter + logs
os.makedirs(WORK_DIR, exist_ok=True)

print("Please upload the labeled fine-tuning file:")
print("  'Announcements manually labeled fine tuning sample.xlsx'")
print("  (columns: 'announcement', 'products')")
uploaded = files.upload()          # opens the file picker

def _resolve(name):
    "files.upload() writes into /content; return an existing path."
    for cand in (name, os.path.join("/content", name)):
        if os.path.exists(cand):
            return cand
    return name

xlsx_files = [f for f in uploaded.keys() if f.lower().endswith((".xlsx", ".xls"))]
assert xlsx_files, "No .xlsx uploaded. Re-run this cell and pick the labeled file."
LABELED_XLSX = _resolve(xlsx_files[0])
print("\nLabeled (fine-tuning) file:", LABELED_XLSX)


## 4. Load labeled data + build stratification categories

The `products` column is stored as a **string** representation of a Python list (e.g. `["ProductA", "ProductB"]` or `[]`). We parse it robustly, then assign each row a category:

| Category | Meaning |
|---|---|
| `none` | `[]` — no product |
| `single` | exactly 1 product |
| `multi` | 2+ products |


In [ ]:
import pandas as pd
import ast

df = pd.read_excel(LABELED_XLSX)
df.columns = [c.strip().lower() for c in df.columns]
assert "announcement" in df.columns and "products" in df.columns, f"Expected columns 'announcement','products'; got {list(df.columns)}"

def parse_products(x):
    "Robustly parse the products cell into a list[str]."
    if isinstance(x, list):
        return [str(p).strip() for p in x if str(p).strip()]
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return []
    try:
        val = ast.literal_eval(s)          # handles '["a","b"]' and '[]'
        if isinstance(val, list):
            return [str(p).strip() for p in val if str(p).strip()]
        return [str(val).strip()]
    except (ValueError, SyntaxError):
        return [s]  # fallback: treat raw string as one product

df["products_list"] = df["products"].apply(parse_products)
df["n_products"] = df["products_list"].apply(len)

def category(n):
    if n == 0: return "none"
    if n == 1: return "single"
    return "multi"

df["strata"] = df["n_products"].apply(category)

print("Total rows:", len(df))
print(df["strata"].value_counts())
df[["announcement","products_list","n_products","strata"]].head()


## 5. Stratified 80 / 10 / 10 split (train / eval / out-of-sample test)

Stratified on the none/single/multi category so each set keeps the same product-count mix. **Train** fits the LoRA, **eval** drives the per-epoch eval loss during training, and **test** is held completely out of sight for a final out-of-sample prediction pass.

In [ ]:
from sklearn.model_selection import train_test_split

# 80 / 10 / 10 stratified split -> train / eval / out-of-sample test.
# Stratify on the none/single/multi category so all three sets keep the same mix.
# Done in two steps: first hold out 20% (eval+test), then split that 20% in half.
TEST_FRACTION = 0.10   # out-of-sample test (never seen in training or eval)
EVAL_FRACTION = 0.10   # used by the Trainer for per-epoch eval loss

train_df, temp_df = train_test_split(
    df,
    test_size=TEST_FRACTION + EVAL_FRACTION,   # 0.20 held out
    random_state=SEED,
    shuffle=True,
    stratify=df["strata"],
)
# split the held-out 20% into eval (10%) and test (10%) -> 0.5 of temp each
eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
    stratify=temp_df["strata"],
)

print(f"Train: {len(train_df)}   Eval: {len(eval_df)}   Test (out-of-sample): {len(test_df)}")
for name, d in [("Train", train_df), ("Eval", eval_df), ("Test", test_df)]:
    print(f"\n{name} strata proportions:")
    print(d["strata"].value_counts(normalize=True).round(3))


## 6. Build prompt-completion training examples (loss on answer only)

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are an expert at extracting product names from announcements about product "
    "launches, introductions, upgrades, enhancements, regulatory approvals.\n"
    "Extract specific product names of the products being launched, introduced, "
    "upgraded, enhanced, or approved. If multiple products in the announcement meet "
    "the criteria, extract these multiple products. Include the exact model or version "
    "of the products if it's given. The product names extracted should have proper "
    "nouns, not just common nouns.\n"
    "Do not extract products that are not being launched, introduced, upgraded, "
    "enhanced, or approved.\n"
    "For each extracted product, include the product's corresponding company "
    "name (e.g. \"Microsoft Windows XP\" instead of just \"Windows XP\"; or "
    "\"Apple iPhone 16\" instead of just \"iPhone 16\"). If the product belongs to "
    "multiple companies, add the respective company names (separated by a slash) to the "
    "product name (e.g. \"Apple / Mercedes-Benz iPod(R) Integration Kit\").\n"
    "If the announcement contains company names but no specific product names, return "
    "an empty array [].\n"
    "Return ONLY a JSON array of strings. If no relevant products are mentioned, "
    "return an empty array []."
)

def make_user_prompt(announcement: str) -> str:
    return (
        "Extract the product names from the following announcement.\n\n"
        f"Announcement:\n{announcement}\n\n"
        "Return only a JSON array of product name strings."
    )

# We use TRL's *prompt-completion* format. With completion_only_loss=True (the
# default for prompt-completion datasets), loss is computed ONLY on the
# completion (the assistant's JSON answer) and the prompt is masked to -100 --
# exactly what Unsloth's train_on_responses_only did, but without needing the
# {% generation %} template markers that Qwen2.5 does not ship.
def to_prompt_completion(announcement: str, products_list):
    target = json.dumps(products_list, ensure_ascii=False)
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": make_user_prompt(announcement)},
    ]
    completion = [
        {"role": "assistant", "content": target},
    ]
    return {"prompt": prompt, "completion": completion}

train_records = [to_prompt_completion(r.announcement, r.products_list) for r in train_df.itertuples()]
eval_records  = [to_prompt_completion(r.announcement, r.products_list) for r in eval_df.itertuples()]

print("Example completion target:", train_records[0]["completion"][0]["content"][:200])


## 7. Load Qwen2.5-7B-Instruct in full fp32 + attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

MODEL_NAME  = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 1536   # longer system prompt (~180 tok) + announcement (~560 tok)
                     # + user wrapper + JSON target can reach ~1150 tok; 1536 gives headroom

# --- FULL fp32, NO quantization ----------------------------------------------
# fp32 is the whole point: no 16-bit rounding means matmul results don't depend
# on GPU reduction order, so losses are bit-reproducible. ~28 GB of weights fits
# easily in 96 GB. We pin the dtype explicitly and use the "eager" attention
# implementation (SDPA/flash paths can be nondeterministic; eager is exact).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype        = torch.float32,   # full fp32 compute
    attn_implementation = "eager",        # deterministic attention (not SDPA/flash)
    device_map         = {"": 0},
)
model.config.use_cache = False            # required with gradient checkpointing
model.gradient_checkpointing_enable()     # 7B fp32 -> save activation memory

# --- LoRA (same hyperparameters as before) -----------------------------------
lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0.0,     # 0 -> deterministic (no dropout RNG)
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\nBase model loaded in", next(model.parameters()).dtype, "- full fp32, no quantization.")


## 8. Wrap examples in a Dataset (prompt-completion format)

In [ ]:
from datasets import Dataset

# Prompt-completion conversational dataset. TRL's SFTTrainer applies the Qwen2.5
# chat template internally and (because each example has a "prompt" key) computes
# loss on the completion tokens only. No manual chat-template call needed.
train_ds = Dataset.from_list(train_records)
eval_ds  = Dataset.from_list(eval_records)

print("train examples:", len(train_ds), "| eval examples:", len(eval_ds))
print("keys:", train_ds.column_names)


## 9. Trainer + per-epoch loss table

Trains for **5 epochs**. `load_best_model_at_end=True` reloads the checkpoint with the **lowest validation (eval) loss** once training finishes, so the out-of-sample test (Section 10) and inference (Section 11) use the best model — not necessarily the last epoch. The per-epoch training + evaluation loss table is printed on screen at the end.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import pandas as pd

EPOCHS = 5
# adamw_torch is the plain (non-fused) optimizer -> deterministic. fp32 states.
OPTIM  = "adamw_torch"

class EpochLossCollector(TrainerCallback):
    "Collects the last train loss and eval loss reported in each epoch."
    def __init__(self):
        self.train_by_epoch = {}
        self.eval_by_epoch  = {}
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        ep = int(round(state.epoch)) if state.epoch else 0
        if "loss" in logs:
            self.train_by_epoch[ep] = logs["loss"]
        if "eval_loss" in logs:
            self.eval_by_epoch[ep] = logs["eval_loss"]

collector = EpochLossCollector()

sft_config = SFTConfig(
    output_dir              = OUTPUT_DIR,
    per_device_train_batch_size = 2,     # fp32 7B on 96 GB -> plenty of room
    gradient_accumulation_steps = 4,     # effective batch size 8
    per_device_eval_batch_size  = 1,     # bs=1 -> no padding effects in eval loss
    warmup_ratio            = 0.05,
    num_train_epochs        = EPOCHS,
    learning_rate           = 1e-4,
    logging_strategy        = "epoch",
    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    save_total_limit        = EPOCHS,          # keep every epoch checkpoint so best can be reloaded
    load_best_model_at_end  = True,            # after training, reload the best checkpoint...
    metric_for_best_model   = "eval_loss",     # ...selected by LOWEST validation loss
    greater_is_better       = False,
    optim                   = OPTIM,
    weight_decay            = 0.01,
    lr_scheduler_type       = "linear",
    seed                    = SEED,
    data_seed               = SEED,      # deterministic shuffling/batching
    full_determinism        = True,      # HF enables torch determinism + disables TF32
    dataset_num_proc        = 1,         # single proc -> deterministic map
    group_by_length         = False,
    dataloader_num_workers  = 0,         # 0 workers -> deterministic order
    report_to               = "none",
    max_length              = MAX_SEQ_LEN,
    # NO bf16 / fp16 -> full fp32 training (both default False, set explicitly):
    bf16                    = False,
    fp16                    = False,
    tf32                    = False,     # TF32 off (nondeterministic rounding)
    # Prompt-completion dataset -> loss on the assistant JSON answer only
    # (prompt tokens masked to -100). Deterministic, no {% generation %} needed.
    completion_only_loss    = True,
    eos_token               = "<|im_end|>",   # align EOS with Qwen chat template
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    args             = sft_config,
    callbacks        = [collector],
)

# Re-seed right before training for a clean, repeatable starting RNG state.
set_all_seeds(SEED, hard=True)
trainer_stats = trainer.train()


# --- Per-epoch training + evaluation loss table (on screen) ------------------
# (Required diagnostic; shown on screen, not saved to disk.)
epochs = sorted(set(collector.train_by_epoch) | set(collector.eval_by_epoch))
loss_table = pd.DataFrame({
    "epoch":           epochs,
    "training_loss":   [collector.train_by_epoch.get(e, float("nan")) for e in epochs],
    "evaluation_loss": [collector.eval_by_epoch.get(e, float("nan"))  for e in epochs],
}).set_index("epoch")

print("\n" + "="*54)
print("  Per-epoch training loss + evaluation loss")
print("="*54)
try:
    from IPython.display import display
    display(
        loss_table.style
        .format({"training_loss": "{:.6f}", "evaluation_loss": "{:.6f}"})
        .set_caption("Per-epoch training vs evaluation loss")
    )
except Exception:
    print(loss_table.round(6).to_string())


## 10. Out-of-sample test predictions (download .xlsx)

Runs the fine-tuned model (greedy, deterministic) over the **held-out test set** (the 10% never used in training or eval) and downloads an Excel file of predictions — the equivalent of the old `eval_predictions` file, but for the out-of-sample test set. No metrics are computed or displayed.

In [ ]:
import json, ast, re
import pandas as pd

# --- Put the model in deterministic greedy eval mode -------------------------
model.eval()
model.config.use_cache = True
# Qwen2.5 ships sampling defaults (temperature/top_p/top_k) that are meaningless
# under greedy decoding; clear them so greedy is unambiguous and warning-free.
gc = model.generation_config
gc.do_sample = False
gc.temperature = None
gc.top_p = None
gc.top_k = None

@torch.no_grad()
def generate_products(announcement: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": make_user_prompt(announcement)},
    ]
    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    input_ids      = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)
    out = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    gen = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def parse_products_out(text: str):
    "Parse a model generation into a list[str]; returns [] on failure."
    if not text:
        return []
    s = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    for parser in (json.loads, ast.literal_eval):
        try:
            v = parser(s)
            if isinstance(v, list):
                return [str(p).strip() for p in v if str(p).strip()]
        except Exception:
            pass
    m = re.search(r"\[.*?\]", s, flags=re.DOTALL)
    if m:
        try:
            v = json.loads(m.group(0))
            if isinstance(v, list):
                return [str(p).strip() for p in v if str(p).strip()]
        except Exception:
            pass
    return []

# --- Run over the out-of-sample TEST set (greedy, deterministic) --------------
set_all_seeds(SEED)
test_preds, test_gold, test_cat = [], [], []
for row in test_df.itertuples():
    raw = generate_products(row.announcement)
    test_preds.append(parse_products_out(raw))
    test_gold.append(list(row.products_list))
    test_cat.append(row.strata)

print("Generated predictions for", len(test_preds), "out-of-sample test announcements.")

# --- Build + download the predictions .xlsx (no metrics) ----------------------
test_out = pd.DataFrame({
    "announcement": test_df["announcement"].values,
    "category":     test_cat,
    "gold":         [json.dumps(g, ensure_ascii=False) for g in test_gold],
    "pred":         [json.dumps(p, ensure_ascii=False) for p in test_preds],
    "n_gold":       [len(g) for g in test_gold],   # number of products in gold
    "n_pred":       [len(p) for p in test_preds],  # number of products in prediction
})
TEST_PRED_XLSX = os.path.join(WORK_DIR, "test_predictions.xlsx")
test_out.to_excel(TEST_PRED_XLSX, index=False)
print("Saved:", TEST_PRED_XLSX)

from google.colab import files as _f
_f.download(TEST_PRED_XLSX)
print("Download started: 'test_predictions.xlsx'")


## 12. Save the fine-tuned LoRA adapter and download it to your PC

Saves the **best checkpoint's** adapter (the one with the lowest validation loss — it's already loaded because `load_best_model_at_end=True`) plus the tokenizer, then zips and downloads it. Expected size **~160 MB** (fp32 adapter). Keep this `.zip`; you upload it in notebook 2. The adapter is saved in the **same fp32 precision** it was trained in, so inference in notebook 2 reproduces exactly.

In [ ]:
# --- Save the trained LoRA adapter + tokenizer, then zip & download ----------
import os, shutil, glob
from google.colab import files as _f

ADAPTER_DIR = "/content/qwen25_7b_lora_adapter"   # what we save + download
os.makedirs(ADAPTER_DIR, exist_ok=True)

# `model` currently holds the BEST checkpoint (load_best_model_at_end=True).
# save_pretrained on a PEFT model writes ONLY the adapter (adapter_model.safetensors
# + adapter_config.json) in the model's current dtype (fp32) — not the base weights.
model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)   # so notebook 2 uses the identical tokenizer

# Show what was written + the adapter size.
print("Saved adapter files:")
total = 0
for f in sorted(glob.glob(os.path.join(ADAPTER_DIR, "*"))):
    sz = os.path.getsize(f)
    total += sz
    print(f"  {os.path.basename(f):32s} {sz/1e6:8.2f} MB")
print(f"  {'TOTAL':32s} {total/1e6:8.2f} MB")

# Record which base model + key LoRA settings this adapter was trained against,
# so notebook 2 can assert they match (guards against silent mismatches).
import json as _json
meta = {
    "base_model":   MODEL_NAME,          # "Qwen/Qwen2.5-7B-Instruct"
    "max_seq_len":  MAX_SEQ_LEN,
    "dtype":        "float32",
    "seed":         SEED,
}
with open(os.path.join(ADAPTER_DIR, "finetune_meta.json"), "w") as fh:
    _json.dump(meta, fh, indent=2)
print("\nMeta:", meta)

# Zip the whole adapter folder into a single file for a clean one-click download.
ZIP_BASE = "/content/qwen25_7b_lora_adapter"          # -> qwen25_7b_lora_adapter.zip
zip_path = shutil.make_archive(ZIP_BASE, "zip", ADAPTER_DIR)
print(f"\nZipped adapter: {zip_path}  ({os.path.getsize(zip_path)/1e6:.2f} MB)")

_f.download(zip_path)
print("\nDownload started: 'qwen25_7b_lora_adapter.zip'")
print("Keep this file — you upload it in NOTEBOOK 2 to run inference without re-training.")
